In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from torch import nn
import tqdm
from pathlib import Path
import torch
from datetime import datetime
device = "cuda" if torch.cuda.is_available() else "cpu"
device
%load_ext autoreload
%autoreload 2

PRC = Price
SHROUT = Outstanding Shares
RET = return
VOL = volume
AT = total assets
GP = gross profits
Sale = sales
Ib = income
SEQ = shareholder's equity

entries with NaN prc also have NaN ret

sp.head()

In [ ]:
def loadData():
    sp_raw = pd.read_csv(
        "Data\StockPrices.csv",
        usecols=["permno", "gvkey", "year", "month", "time", "ret", "vol", "prc", 'shrout']   # only load what you need
    )
    ff_raw = pd.read_csv("Data\F-F_Research_Data_Factors.csv")
  
    ffm_raw = pd.read_csv("Data\F-F_Momentum_Factor.csv") 
    
    fund_raw = pd.read_csv(
        "Data\FirmFundamentals.csv",
        usecols=["gvkey", "datadate", "epspx"]   # only load what you need
    )
    
    return sp_raw,ff_raw,ffm_raw, fund_raw
sp_raw, ff_raw, ffm_raw, fund_raw = loadData()

In [ ]:
#DEFINING FUNCTIONS
def Clean_Side_Data(ff_raw: pd.DataFrame, ffm_raw: pd.DataFrame, fund_raw: pd.DataFrame):
    ff = ff_raw.copy()
    ffm = ffm_raw.copy()
    fund = fund_raw.copy()

    ff['Mkt-RF'] = ff['Mkt-RF']/100
    ff['RF'] = ff['RF']/100
    ff['SMB'] = ff['SMB']/100
    ff['HML'] = ff['HML']/100
    #convert from percentages to decimals
    #ff stands for FAMA french
    ffm = ffm.set_index('date')
    ffm.index = pd.to_datetime(ffm.index.astype(str), format="%Y%m") + pd.offsets.MonthEnd(0)
    #converts ffm to date time
    ff = ff.set_index('date')
    ff.index = pd.to_datetime(ff.index.astype(str), format="%Y%m") + pd.offsets.MonthEnd(0)
    #also turns into proper datetime format

    #merges the two together

    ff['Mom'] = ff.index.map(ffm['Mom'])

    ff['Mom'] = ff['Mom']/100

    
    
    lower = fund["epspx"].quantile(0.01)
    upper = fund["epspx"].quantile(0.99)
    fund["gvkey"] = fund["gvkey"].astype("Int64")
    fund["eps_clean"] = fund["epspx"].clip(lower, upper)
   
    fund['datadate'] = pd.to_datetime(fund['datadate'], format="%d%b%Y") + pd.offsets.MonthEnd(0)
    return ff, ffm, fund

def initialize(sp_raw: pd.DataFrame,ff: pd.DataFrame):
    sp = sp_raw.copy()
    
    #sp = sp.dropna(subset = "ret")
    #removes rows where return is NaN
    sp["gvkey"] = sp["gvkey"].astype("Int64")
    sp['date'] = sp['year']*100 + sp['month']
    sp['date'] = pd.to_datetime(sp["date"].astype(str), format="%Y%m") + pd.offsets.MonthEnd(0)
    sp['Mktcap'] = sp['shrout']*sp['prc']
    sp["Mktcap_lag"] = sp.groupby("permno")["Mktcap"].shift(1)
    #sp = sp[sp['Mktcap'] >= 2000]
    sp['Mkt-RF'] = sp['date'].map(ff['Mkt-RF'])
    sp['RF'] = sp['date'].map(ff['RF'])

    sp["gross_ret"] = 1 + sp["ret"]  # ret is monthly return (not price change)
    # For each permno, the 12-1 momentum is product of gross_ret from t-12 .. t-2 minus 1,
    # which equals rolling product of length 11 ending at shift(2)

    #1 year momentum, centered at 0
    sp["mom_12_1 lag"] = (
        sp.groupby("permno")["gross_ret"].transform(lambda s: s.shift(2).rolling(11, min_periods=11).apply(np.prod, raw=True) - 1)
    )

    #normalized 1 year SMA, centered at 0
    sp["norm_SMA_12 lag"] = sp.groupby("permno")['prc'].transform(lambda x: x.shift(1)/(x.shift(1).rolling(12,min_periods=12).mean())-1)

    #liquidity
    sp["turnover"] = sp["vol"] / sp["shrout"]
    sp["turnover lag"] = sp.groupby("permno")['turnover'].shift(1)
    sp['ret_Demeaned'] = sp['ret'] - sp.groupby('date')['ret'].transform('mean')
    #volatility
    sp["volatility_12 lag"] = sp.groupby("permno")["ret"].transform(
    lambda x: x.shift(1).rolling(12).std())

    

    

    sp_merge = pd.merge_asof(sp.sort_values(['date', 'gvkey']), fund.sort_values(["datadate", "gvkey"]), left_on="date", right_on="datadate", by="gvkey", direction="backward" )
    #adds the eps_clean column, as well as many others, returning MANY NaN VALUES --> HIGH NAN activity
    sp_merge = sp_merge.sort_values(["gvkey", "time"])
    #always do this

    
    sp_merge['E/P'] = sp_merge['eps_clean']/sp_merge['prc']
    sp_merge['E/P lag'] = sp_merge.groupby("permno")['E/P'].shift(1)
    
    return sp_merge


def backtest(top_pct: float, bot_pct: float, sp: pd.DataFrame = None, rank_name: str = None, model: nn.Module =None, backtest_dataset: Dataset = None):
  #IF WE ARE USING A MODEL, leave sp and rank_name blank
  
    if model:
        df = backtest_dataset.df.copy()
        target_params = backtest_dataset.target_params
        model.eval()
        with torch.no_grad():
            X_test = df[target_params].values.astype("float32")
            df['pred'] = model(torch.tensor(X_test).to(device)).cpu().numpy()        
            df['rank_percentile'] = df.groupby('date')['pred'].rank(pct=True)

    elif sp:
        df = sp.copy()
        df["rank_percentile"] = (
            df.groupby("date")[rank_name]
                .rank(pct=True)
    )
    else:
        raise Exception("Must need either sp or model parameters!")
    
    top = df[df["rank_percentile"] >= top_pct].copy()
    bot = df[df["rank_percentile"] <= bot_pct].copy()

    #array in time called portfolio gives return at each point in time
    print("avg no. of stocks in top " + str(top.groupby('date')['ret'].count().mean()))
    print("avg no. of stocks in bottom: " + str(bot.groupby('date')['ret'].count().mean()))
    top["side"] = "long"
    bot["side"] = "short"
    
    pdf = pd.concat([top, bot]).copy()
    
    #portfolio_df = pd.concat([top, bot])
    #pdf = portfolio_df.copy().dropna(subset=["ret", "Mktcap"])

    # weight within each (date, side) group, so each group's weights sum to 1
    pdf["w"] = pdf.groupby(["date", "side"])["Mktcap_lag"].transform(lambda x: x / x.sum())
    #mkt cap_lag was very important
    # weighted return per row, then sum by group
    pdf["w_ret"] = pdf["ret"] * pdf["w"]

    portfolio_returns = pdf.groupby(["date", "side"])["w_ret"].sum().unstack()
    portfolio_returns["strategy"] = (portfolio_returns['long'] - portfolio_returns["short"])
    portfolio_returns['Mkt-RF'] = portfolio_returns.index.map(ff['Mkt-RF'])
    portfolio_returns['RF'] = portfolio_returns.index.map(ff['RF'])
    return portfolio_returns
    


In [ ]:
#ENTIRE CODE IS HERE (Except loading the data)
ff, ffm, fund = Clean_Side_Data(ff_raw, ffm_raw, fund_raw)
sp = initialize(sp_raw,ff)


In [ ]:
class FinancialDataset(Dataset):
    def __init__(self, target_params: list, sp: pd.DataFrame):
        df = sp.copy()
        df = df[["date", "permno", "ret", "Mktcap_lag", "Mkt-RF", 'ret_Demeaned'] + target_params]
        df = df.dropna()
        for param in target_params:
            df[param] = df.groupby("date")[param].rank(pct=True)
            #normalizes
        df['index'] = df['date'].rank(method='min')
        self.df = df
        self.target_params = target_params
        
        
        
        
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        series_x, series_y = self.df.iloc[index][self.target_params], self.df.iloc[index]['ret_Demeaned']
        x = torch.tensor(series_x.values.astype("float32"))
        y = torch.tensor(series_y.astype("float32"))
        return x, y
        


In [ ]:
target_params = ["mom_12_1 lag", "norm_SMA_12 lag", "turnover lag", "volatility_12 lag", "E/P lag"]
separating_date = '2018-01-01'
test_sp = sp[sp['date'] >= separating_date].copy()
train_sp = sp[sp['date']  < separating_date].copy()
train_dataset = FinancialDataset(target_params, train_sp)
test_dataset = FinancialDataset(target_params, test_sp)

In [ ]:
batch_size = 1024
hidden_units = [64,32,16]
if device == 'cuda':
    num_workers = 0
else:
    num_workers = 0
#in jupyter notebook you cant use num_workers > 0  sadly
train_dataloader = DataLoader(dataset=train_dataset,batch_size=batch_size,shuffle = True, num_workers=num_workers, pin_memory= True)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=batch_size,shuffle=False,num_workers=num_workers, pin_memory = True)
class StockPredictor(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_features=input_features, out_features=hidden_units[0]),
            nn.ReLU(),
            nn.Linear(hidden_units[0],hidden_units[1]),
            nn.ReLU(),
            nn.Linear(hidden_units[1],hidden_units[2]),
            nn.ReLU(),
            nn.Linear(hidden_units[2],output_features)
        )
    def forward(self,x):
        return self.block(x)

model = StockPredictor(input_features=len(target_params),output_features=1).to(device)
loss_fn = nn.HuberLoss()
lr = 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

In [ ]:


def TrainModel(model, epochs, train_dataloader, loss_fn, optimizer):
    model.train()
    batch_loss = 0
    for epoch in range(epochs):
        
        for batch, (x_train, y_train) in enumerate(train_dataloader):
            
            x_train, y_train = x_train.to(device, non_blocking = True), y_train.to(device, non_blocking = True)
            y_pred = model(x_train).squeeze()
            
            loss = loss_fn(y_pred,y_train)

            optimizer.zero_grad()

            loss.backward()

            
            optimizer.step()
            batch_loss += loss.item()
            if batch % 100 == 0:
                print(f"at epoch {epoch},batch {batch}, the average loss is {batch_loss/100}")
                batch_loss = 0
                
epochs = 2
TrainModel(model, epochs, train_dataloader, loss_fn, optimizer)


Training with batch size = 2048 took 12 minutes and 47s, while with size = 256 it took about 14 minutes.
With pin_memory and non_blocking, 12 mins and 51 seconds, literally the same bruh

batch size 1024 took 12 min 38 seconds, odd.

In [ ]:
def EvaluateModel(model: nn.Module, top_pct, bot_pct, backtest_dataset):
    train_returns = backtest(top_pct, bot_pct,model = model, backtest_dataset=train_dataset)
    test_returns = backtest(top_pct, bot_pct,model = model, backtest_dataset=test_dataset)
    total_returns = backtest(top_pct, bot_pct, model = model, backtest_dataset = backtest_dataset)

    sharpes = []
    returns = []
    top_pct = top_pct *np.ones(3)
    bot_pct = bot_pct *np.ones(3)
    #first index is train, second index test, third index total 
    for df in [train_returns, test_returns, total_returns]:
        sharpes.append(np.sqrt(12)*(df["strategy"]-df["RF"]).mean()/(df["strategy"]-df['RF']).std())
        returns.append(df["strategy"].mean()*12)
    output = pd.DataFrame([sharpes, returns,top_pct, bot_pct], columns=["Train Dataset", "Test Dataset", "Whole Dataset"], index = ['Sharpe Ratio', 'Average Yearly Return', 'Top %', 'Bottom %'])



    return output
def DeepEvaluateModel(model: nn.Module, top_pct, bot_pct, sp: pd.DataFrame, dateslice: list):
    sp_slice = sp[sp['date'] >= dateslice[0] and sp['date'] <= dateslice[1]].copy()
    portfolio_returns = backtest(top_pct, bot_pct, model = model, backtest_dataset = sp_slice)
    plt.plot(portfolio_returns["strategy"])
    plt.title('Graph of strategy returns against time')
    plt.show()

    sharpe = (portfolio_returns["strategy"]-portfolio_returns["RF"]).mean()/(portfolio_returns["strategy"]-portfolio_returns['RF']).std()

    print("yearly sharpe ratio (adjusted for risk free) = " + str(sharpe*np.sqrt(12)))
    print("mean return per month: "  + str(portfolio_returns["strategy"].mean()))
    print("mean return per year: "  + str(portfolio_returns["strategy"].mean()*12))

    portfolio_returns = portfolio_returns.dropna(subset = ["strategy", "RF", "Mkt-RF", "Mom"])
    portfolio_returns["HML"] = portfolio_returns.index.map(ff['HML'])
    portfolio_returns["Mom"] = portfolio_returns.index.map(ff['Mom'])

    y = portfolio_returns["strategy"] - portfolio_returns["RF"]  #the y is the adjusted return
    X = portfolio_returns[["Mkt-RF",'Mom']]
    X = sm.add_constant(X) #this is the constant term which allows us to find alpha
    OLSmodel = sm.OLS(y, X).fit()

    #run OLS
    print(OLSmodel.summary())
    plt.scatter(X['Mkt-RF'],y)
    test_x = np.linspace(-0.3,0.3,1000)
    test_y = test_x*(OLSmodel.params["Mkt-RF"]) + OLSmodel.params['const']

    plt.plot(test_x, test_y, color = 'r')
    plt.xlabel("Mkt-RF")
    plt.ylabel("Strategy return")
    plt.title("Plot of strategy return against market return only")
    plt.show() 
    print("Note that I may use OLS accounting for several factors")
    print("so the gradient of the line of best fit is dependent on what kind of factors i have added")
    #plots the data and line of best fit

backtest_dataset = FinancialDataset(target_params=target_params,sp=sp)
results = EvaluateModel(model, 0.9,0.1, backtest_dataset)
print(results)

    




In [ ]:

def SaveModel(model: nn.Module, model_name: str = None, comments: str = None):
    
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    # save model weights
    

    if not model_name:
        model_name = timestamp
    if not comments:
        comments = 'No comment'
    # save benchmarks / metrics
    save_dir = Path("Models/" + model_name)
    
    if save_dir.exists():
        raise FileExistsError(f"{save_dir} already exists")
    save_dir.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), save_dir / "model.pt")
    results_md = results.to_markdown()

    summary_md = f"""
    #Model Name: {model_name} 

    Created: {timestamp}

    ## Config
    - Epochs: {epochs}
    - Batch size: {batch_size}
    - Learning rate: {lr}
    - Hidden Units: {hidden_units}
    - Target Parameters: {target_params}

    ## Notes
    {comments}

    #Performance: 
    """
    summary_md += results_md

    with open(Path(str(save_dir) + "/summary.md"), "w") as f:
        f.write(summary_md)
SaveModel(model, model_name='deep layer', comments='so i used a deeper neural network with layernorm and dropout, seems to have little effect')
    # save config too, if useful